# Motor Insurance Fraud Detection
## Final Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ktnspr/py4lju/blob/main/09_fraud_detection_exercise.ipynb)

Build a classifier that detects fraudulent motor insurance claims. Work through three phases:

- **Phase 1**: Exploration
- **Phase 2**: Preparation and Modelling
- **Phase 3**: Evaluation

Phases 1 and 2 repeat steps from the exercises in Notebooks 05 and 07 on purpose: here they come together in one model. Your code from there can serve as a starting point.

### The Dataset: `vehicle_fraud.csv`

10,050 rows of synthetic motor insurance claims:

| Column | Description |
|--------|-------------|
| `claim_id` | Claim number |
| `policyholder_age` | Age in years |
| `occupation` | `Employed`, `Self-employed`, `Unemployed`, `Student` |
| `region` | Region `R01` to `R10` |
| `credit_score` | Credit score (300–850) |
| `customer_since_years` | Years as a customer |
| `vehicle_value` | Vehicle value |
| `vehicle_type` | `Compact`, `Mid-range`, `Luxury`, `SUV`, `Sports car` |
| `coverage_type` | `Liability`, `Partial coverage`, `Comprehensive` |
| `number_of_prior_claims` | Number of earlier claims |
| `claim_amount` | Claim amount |
| `police_report` | `1` if a police report was filed |
| `witnesses_present` | `1` if witnesses were present |
| `reporting_delay_days` | Days between accident and claim report |
| `fraud` | Target: `1` = fraudulent, `0` = legitimate |

In [ ]:
# Colab setup: fetch the dataset if not running locally
import sys
if 'google.colab' in sys.modules:
    !wget -qN https://raw.githubusercontent.com/ktnspr/py4lju/main/vehicle_fraud.csv

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, precision_recall_curve)
from sklearn.inspection import permutation_importance

# Seaborn style for all plots
sns.set_theme(style='whitegrid')

# Avoid truncation of DataFrame output in Jupyter
pd.set_option('display.expand_frame_repr', False)

# Display all columns in the DataFrame output
pd.set_option('display.max_columns', None)

# Define a separator for better readability
sep = "=" * 50

In [ ]:
# Load the dataset
df = pd.read_csv('vehicle_fraud.csv')
df.head()

## Phase 1: Exploration

Get to know the data: size, missing values, fraud rate, distributions and the first fraud patterns.

This exploration is deliberately short and focuses on fraud. The general exploration of the data took place in Notebook 06.

### 1.1 Overview

**Task 1:** How many rows and columns does the dataset have? Which columns are numeric, which are text?

In [ ]:
# Task 1: Shape and data types
# Hint: df.shape, df.info()


**Task 2:** Are there missing values?

In [ ]:
# Task 2: Missing values
# Hint: df.isna().sum()


**Task 3:** What is the fraud rate?

In [ ]:
# Task 3: Fraud rate
# Hint: df['fraud'].value_counts(normalize=True)


### 1.2 Distributions

**Task 1:** Plot a histogram of `claim_amount`. Is it symmetric or skewed?

In [ ]:
# Task 1: Histogram of claim_amount
# Hint: df['claim_amount'].plot(kind='hist', bins=50)
#       plt.title('...'), plt.xlabel('...'), plt.ylabel('...')
#       plt.show()


**Task 2:** Compare `claim_amount` for fraudulent and legitimate claims with a box plot.

In [ ]:
# Task 2: Box plot of claim_amount by fraud
# Hint: sns.boxplot(data=df, x='fraud', y='claim_amount')


### 1.3 Fraud Patterns

**Task 1:** Compare the mean of every numeric column for `fraud == 0` and `fraud == 1`.

In [ ]:
# Task 1: Means by fraud status
# Hint: df.groupby('fraud').mean(numeric_only=True)


**Task 2:** Compute the correlation of every numeric column with `fraud` and sort it.

In [ ]:
# Task 2: Correlation with fraud
# Hint: df.corr(numeric_only=True)['fraud'].sort_values()


**Task 3:** Compute the fraud rate for each combination of `police_report` and `witnesses_present`.

In [ ]:
# Task 3: Fraud rate by documentation
# Hint: df.groupby(['police_report', 'witnesses_present'])['fraud'].mean()


**Task 4:** Compute the fraud rate for new customers (`customer_since_years < 3`) with many prior claims (`number_of_prior_claims >= 3`).

In [ ]:
# Task 4: Fraud rate of new customers with many prior claims
# Hint: new_many = (df['customer_since_years'] < 3) & (df['number_of_prior_claims'] >= 3)
#       df.groupby(new_many)['fraud'].mean()


## Phase 2: Preparation and Modelling

Apply Notebooks 05 and 07: clean, engineer features, encode, split, fill, scale and train one or more models.

### 2.1 Cleaning

**Task 1:** How many rows are duplicates? Remove them.

In [ ]:
# Task 1: Duplicates
# Hint: df.duplicated().sum()
#       df = df.drop_duplicates()


**Task 2:** Find the impossible ages with `describe()` and replace them with `np.nan`.

In [ ]:
# Task 2: Impossible ages
# Hint: df['policyholder_age'].describe()
#       df.loc[df['policyholder_age'] < 18, 'policyholder_age'] = np.nan


**Task 3:** Add the indicator column `credit_score_missing`. Compare the fraud rate of claims with and without `credit_score`.

In [ ]:
# Task 3: Indicator column and fraud rate
# Hint: df['credit_score_missing'] = df['credit_score'].isna().astype(int)
#       df.groupby('credit_score_missing')['fraud'].mean()


### 2.2 Feature Engineering

Create four new columns:

**Task 1:** `claim_value_ratio` = `claim_amount` / `vehicle_value`

In [ ]:
# Task 1: Claim-to-value ratio
# Hint: df['claim_value_ratio'] = df['claim_amount'] / df['vehicle_value']


**Task 2:** `no_report_no_witness` = `1` if there is neither a police report nor witnesses

In [ ]:
# Task 2: Neither police report nor witnesses
# Hint: ((df['police_report'] == 0) & (df['witnesses_present'] == 0)).astype(int)


**Task 3:** `new_customer_many_claims` = `1` for customers with `customer_since_years < 3` and `number_of_prior_claims >= 3`

In [ ]:
# Task 3: New customer with many prior claims


**Task 4:** `log_claim_amount` = `np.log(claim_amount)` for the skewed claim amounts

In [ ]:
# Task 4: Log-transformed claim amount


Then compare the mean `claim_value_ratio` for fraudulent and legitimate claims.

In [ ]:
# Mean claim_value_ratio by fraud status
# Hint: df.groupby('fraud')['claim_value_ratio'].mean()


### 2.3 Encoding, Split, Missing Values and Scaling

**Task 1:** One-hot encode `occupation`, `region`, `vehicle_type` and `coverage_type` with `pd.get_dummies()`. Store the result in `df_encoded`.

In [ ]:
# Task 1: One-hot encoding
# Hint: pd.get_dummies(df, columns=[...], drop_first=True, dtype=int)


**Task 2:** Split into features `X` (without `claim_id` and `fraud`) and target `y`.

In [ ]:
# Task 2: Features and target
# Hint: X = df_encoded.drop(columns=['claim_id', 'fraud'])
#       y = df_encoded['fraud']


**Task 3:** Split into 80% training and 20% test data. `stratify=y` keeps the fraud rate equal in both sets.

In [ ]:
# Task 3: Train/test split
# Hint: train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


**Task 4:** Fill the missing values in both sets with the medians of the training data.

In [ ]:
# Task 4: Fill missing values with the training medians
# Hint: medians = X_train.median()
#       X_train = X_train.fillna(medians)
#       X_test = X_test.fillna(medians)


**Task 5:** Scale the features with `StandardScaler`. Fit the scaler on the training data only.

In [ ]:
# Task 5: Scaling
# Hint: scaler = StandardScaler()
#       X_train_scaled = scaler.fit_transform(X_train)
#       X_test_scaled = scaler.transform(X_test)


### 2.4 Training

Train one or more of the following classifiers.

With only 12% fraud, a model mostly learns the legitimate claims. Class weighting counteracts this: each fraud case counts as much as about 7.5 legitimate claims (number of legitimate / number of fraudulent claims), so both classes carry the same total weight. `class_weight='balanced'` and `scale_pos_weight` do the same thing; `XGBClassifier` just calls it differently. `MLPClassifier` has no class weighting.

| Model | Input | Class weighting |
|-------|-------|-----------------|
| `LogisticRegression(max_iter=1000)` | scaled | `class_weight='balanced'` |
| `RandomForestClassifier(n_estimators=100)` | unscaled | `class_weight='balanced'` |
| `XGBClassifier(n_estimators=100)` | unscaled | `scale_pos_weight` = number of legitimate / number of fraudulent claims |
| `MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, early_stopping=True)` | scaled | none |

Use `random_state=42` for every model.

In [ ]:
# Train one or more models
# Hint: lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
#       lr_model.fit(X_train_scaled, y_train)
# Hint: for XGBoost, scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

## Phase 3: Evaluation

Compare the models with metrics, find the important features and pick the cheapest threshold.

### 3.1 Metrics

| Metric | Question |
|--------|----------|
| Precision | Of the flagged claims, how many are fraud? |
| Recall | Of all fraud cases, how many are flagged? |
| F1-score | Balance of precision and recall |
| ROC-AUC | How well does the model rank fraud above non-fraud? |

Store your trained models with their test data in the dictionary `models`. Keep only the entries of the models you trained.

In [ ]:
# Uncomment the entries of the models you trained
models = {
    # 'Logistic Regression': (lr_model, X_test_scaled),
    # 'Random Forest': (rf_model, X_test),
    # 'XGBoost': (xgb_model, X_test),
    # 'Neural Network': (mlp_model, X_test_scaled),
}

**Task 1:** For each model, compute accuracy, precision, recall, F1-score and ROC-AUC. Show the results as a DataFrame.

In [ ]:
# Task 1: Metrics table
# Hint: results = {}
#       for name, (model, X_eval) in models.items():
#           y_pred = model.predict(X_eval)
#           y_proba = model.predict_proba(X_eval)[:, 1]
#           results[name] = {'Accuracy': accuracy_score(y_test, y_pred), ...,
#                            'ROC-AUC': roc_auc_score(y_test, y_proba)}
#       results_df = pd.DataFrame(results).T


**Task 2:** Plot a confusion matrix for each model.

In [ ]:
# Task 2: Confusion matrices
# Hint: for name, (model, X_eval) in models.items():
#           cm = confusion_matrix(y_test, model.predict(X_eval))
#           sns.heatmap(data=cm, annot=True, fmt='d', cmap='Blues')
#           plt.title(name), plt.xlabel('Predicted'), plt.ylabel('Actual')
#           plt.show()
#       fmt='d' shows whole numbers


### 3.2 Feature Importance

`permutation_importance()` shuffles one feature at a time in the test data and measures how much the ROC-AUC drops. A large drop means the model relies on that feature.

**Task 1:** Pick one model from `models` and compute its permutation importance.

In [ ]:
# Task 1: Permutation importance
# Hint: name = 'Random Forest'
#       model, X_eval = models[name]
#       perm = permutation_importance(model, X_eval, y_test, n_repeats=10,
#                                     scoring='roc_auc', random_state=42)


**Task 2:** Plot the ten most important features as a horizontal bar chart.

In [ ]:
# Task 2: Top 10 features
# Hint: importances = pd.Series(perm.importances_mean, index=X_test.columns)
#       top10 = importances.sort_values().tail(10)
#       top10.plot(kind='barh')


**Task 3:** Do your new columns from 2.1 and 2.2 appear among them?

### 3.3 Threshold Tuning

`predict()` flags a claim as fraud if its predicted probability is at least 0.5. A lower threshold catches more fraud (higher recall) but causes more false alarms (lower precision). The best threshold depends on the costs:

- Every flagged claim is investigated. An investigation costs 500 €.
- Every missed fraud case is paid out. It costs its `claim_amount`.

$$\text{Cost}(t) = 500 \cdot \text{number of flagged claims} + \text{sum of claim amounts of missed fraud cases}$$

**Task 1:** For the model from 3.2, compute precision and recall for every threshold with `precision_recall_curve()` and plot recall against precision.

In [ ]:
# Task 1: Precision-recall curve
# Hint: y_proba = model.predict_proba(X_eval)[:, 1]
#       precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
#       pr_curve = pd.Series(precisions, index=recalls)
#       pr_curve.plot(kind='line')


**Task 2:** Compute the total cost for the thresholds 0.01, 0.02, ..., 0.99 and plot cost against threshold. Which threshold has the lowest cost? Compare it with 0.5.

In [ ]:
# Task 2: Total cost per threshold
# Hint: def cost_table(investigation_cost):
#           rows = []
#           for t in np.arange(1, 100) / 100:
#               flagged = y_proba >= t
#               missed = ~flagged & (y_test == 1)
#               cost = investigation_cost * flagged.sum() + X_test.loc[missed, 'claim_amount'].sum()
#               rows.append({'threshold': t, 'flagged': flagged.sum(),
#                            'recall': recall_score(y_test, flagged), 'cost': cost})
#           return pd.DataFrame(rows)
# Hint: cost_curve = costs.set_index('threshold')['cost']
#       cost_curve.plot(kind='line')
#       costs.loc[costs['cost'].idxmin()]


**Task 3:** Repeat Task 2 with investigation costs of 200 € and 3000 €. How does the best threshold move?

In [ ]:
# Task 3: Investigation costs of 200 € and 3000 €


## Summary

- **Exploration:** fraud rate, skewed distributions, fraud rates by group
- **Preparation and modelling:** duplicates, impossible values, indicator for missing values, feature engineering, one-hot encoding, stratified split, filling missing values, scaling, class weights
- **Evaluation:** precision, recall, F1-score, ROC-AUC, confusion matrix, permutation importance, cost-based threshold tuning

- [scikit-learn User Guide](https://scikit-learn.org/stable/user_guide.html)
- [XGBoost Documentation](https://xgboost.readthedocs.io/)